# 06 - Treat Service Funding

## Input
- `data/raw/service_list/` — 7 annual snapshots (2019–2025)

## Output
- `data/clean/service_funding_by_facility.csv` — one row per facility × year

## Columns
- `service_name`, `sa3_code`, `sa3_name`, `state`
- `care_type`, `org_type`
- `year`, `funding`

## Note
- Funding = Australian Government funding paid to the facility in that financial year
- Negative values = clawback/reconciliation adjustments (kept as-is)
- 2019–2022 files resolved to SA3 via two-tier postcode + ACPR lookup (same as notebook 02)

In [1]:
import pandas as pd
import numpy as np
import os

RAW = '../../data/raw/service_list'
OUT = '../../data/clean/service_funding_by_facility.csv'

files = sorted([f for f in os.listdir(RAW) if not f.startswith('~')])
POSTCODE_COL = {2023: 'Physical Post Code', 2024: 'Postal Code', 2025: 'Physical Post Code'}

ORG_TYPE_MAP = {
    'private incorporated body': 'profit',
    'publicly listed company':   'profit',
    'charitable':                'not_for_profit',
    'religious':                 'not_for_profit',
    'community based':           'not_for_profit',
    'religious/charitable':      'not_for_profit',
    'state government':          'government',
    'local government':          'government',
    'territory government':      'government',
}

def read_service_list(filepath):
    probe = pd.read_excel(filepath, header=None, nrows=6, engine='calamine')
    hdr_row = next(i for i, r in probe.iterrows() if 'Service Name' in r.values)
    return pd.read_excel(filepath, header=hdr_row, engine='calamine')

def find_funding_col(df):
    return next((c for c in df.columns if 'funding' in str(c).lower()), None)

In [2]:
# =============================================================================
# STEP 1 — Build postcode -> SA3 lookup from 2023-2025 (same as notebook 02)
# =============================================================================

mapping_frames = []
for year in [2023, 2024, 2025]:
    fname = next(f for f in files if str(year) in f)
    df = read_service_list(f'{RAW}/{fname}')
    sa3_col      = next(c for c in df.columns if 'SA3 Code' in str(c))
    sa3_name_col = next(c for c in df.columns if 'SA3 Name' in str(c))
    acpr_col     = next(c for c in df.columns if 'ACPR' in str(c) or 'Planning Region' in str(c))
    tmp = df[[POSTCODE_COL[year], acpr_col, sa3_col, sa3_name_col]].dropna()
    tmp.columns = ['postcode', 'acpr', 'sa3_code', 'sa3_name']
    tmp['postcode'] = tmp['postcode'].astype(str).str.strip().str.split('.').str[0]
    mapping_frames.append(tmp)

raw_mapping = pd.concat(mapping_frames, ignore_index=True)

pc_sa3_counts = raw_mapping.groupby('postcode')['sa3_code'].nunique()
unambiguous   = set(pc_sa3_counts[pc_sa3_counts == 1].index)
ambiguous     = set(pc_sa3_counts[pc_sa3_counts > 1].index)

postcode_sa3_simple = (
    raw_mapping[raw_mapping['postcode'].isin(unambiguous)]
    .drop_duplicates('postcode')[['postcode', 'sa3_code', 'sa3_name']]
)
postcode_acpr_sa3 = (
    raw_mapping[raw_mapping['postcode'].isin(ambiguous)]
    .groupby(['postcode', 'acpr', 'sa3_code', 'sa3_name'])
    .size().reset_index(name='freq')
    .sort_values('freq', ascending=False)
    .drop_duplicates(['postcode', 'acpr'])
    [['postcode', 'acpr', 'sa3_code', 'sa3_name']]
)
print(f'Lookup built: {len(postcode_sa3_simple)} unambiguous, {len(postcode_acpr_sa3)} ambiguous postcodes')

Lookup built: 1361 unambiguous, 222 ambiguous postcodes


In [3]:
# =============================================================================
# STEP 2 — Extract service name, region, year, funding from each file
# =============================================================================

year_file_map = {yr: next(f for f in files if str(yr) in f) for yr in range(2019, 2026)}
frames = []

for year, fname in sorted(year_file_map.items()):
    df = read_service_list(f'{RAW}/{fname}')

    fund_col = find_funding_col(df)
    if not fund_col:
        print(f'{year}: no funding column — skip')
        continue

    has_sa3 = any('SA3 Code' in str(c) for c in df.columns)

    if has_sa3:
        sa3_col      = next(c for c in df.columns if 'SA3 Code' in str(c))
        sa3_name_col = next(c for c in df.columns if 'SA3 Name' in str(c))
        df = df.rename(columns={sa3_col: 'sa3_code', sa3_name_col: 'sa3_name'})
    else:
        acpr_col = next(c for c in df.columns if 'ACPR' in str(c) or 'Planning Region' in str(c))
        df['postcode'] = df['Physical Address Post Code'].astype(str).str.strip().str.split('.').str[0]
        df = df.rename(columns={acpr_col: 'acpr'})
        df = df.merge(postcode_sa3_simple, on='postcode', how='left')
        mask = df['sa3_code'].isna()
        if mask.sum() > 0:
            fill = df.loc[mask, ['postcode', 'acpr']].merge(
                postcode_acpr_sa3, on=['postcode', 'acpr'], how='left'
            )
            df.loc[mask, 'sa3_code'] = fill['sa3_code'].values
            df.loc[mask, 'sa3_name'] = fill['sa3_name'].values

    state_col = next((c for c in df.columns if c in {'Physical State', 'State', 'STATE'}), None)
    care_col  = next((c for c in df.columns if 'Care Type' in str(c)), None)
    org_col   = next((c for c in df.columns if 'Organisation' in str(c)), None)

    out = pd.DataFrame()
    out['service_name'] = df['Service Name']
    out['sa3_code']     = df['sa3_code']
    out['sa3_name']     = df['sa3_name']
    out['state']        = df[state_col] if state_col else np.nan
    out['care_type']    = df[care_col]  if care_col  else np.nan
    out['org_type']     = (df[org_col].astype(str).str.strip().str.lower().map(ORG_TYPE_MAP)
                           if org_col else np.nan)
    out['year']         = year
    out['funding']      = pd.to_numeric(df[fund_col], errors='coerce')

    out = out.dropna(subset=['sa3_code', 'funding'])
    frames.append(out)
    print(f'{year}: {len(out)} facilities | total: ${out["funding"].sum():,.0f} | unmapped org: {out["org_type"].isna().sum()}')

result = pd.concat(frames, ignore_index=True)
print(f'\nCombined: {result.shape}')
print('Years:', sorted(result['year'].unique()))
print('org_type values:', result['org_type'].value_counts().to_dict())

2019: 5473 facilities | total: $15,943,648,131 | unmapped org: 0


2020: 5437 facilities | total: $17,249,311,684 | unmapped org: 0


2021: 5734 facilities | total: $18,754,277,119 | unmapped org: 0


2022: 5522 facilities | total: $20,315,662,457 | unmapped org: 0


2023: 5523 facilities | total: $23,494,323,265 | unmapped org: 0


2024: 5414 facilities | total: $30,531,855,457 | unmapped org: 0
2025: 5378 facilities | total: $34,535,125,499 | unmapped org: 0

Combined: (38481, 8)
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
org_type values: {'not_for_profit': 23121, 'profit': 10752, 'government': 4608}


In [4]:
# =============================================================================
# STEP 3 — Quick sanity check
# =============================================================================

print('=== National total funding by year ===')
print(result.groupby('year')['funding'].sum().apply(lambda x: f'${x:,.0f}').to_string())

print('\n=== Funding by org type (2024) ===')
yr2024 = result[result['year'] == 2024]
print(yr2024.groupby('org_type')['funding'].agg(['sum', 'mean', 'count'])
      .sort_values('sum', ascending=False).to_string())

print('\n=== Top 10 facilities by funding (2024) ===')
print(yr2024.nlargest(10, 'funding')[['service_name', 'sa3_name', 'state', 'org_type', 'funding']].to_string(index=False))

=== National total funding by year ===
year
2019    $15,943,648,131
2020    $17,249,311,684
2021    $18,754,277,119
2022    $20,315,662,457
2023    $23,494,323,265
2024    $30,531,855,457
2025    $34,535,125,499

=== Funding by org type (2024) ===
                         sum          mean  count
org_type                                         
not_for_profit  1.721784e+10  5.405914e+06   3185
profit          1.154652e+10  7.207564e+06   1602
government      1.767503e+09  2.818983e+06    627

=== Top 10 facilities by funding (2024) ===
                                           service_name                 sa3_name state org_type      funding
                                           Trilogy Care   Brisbane Inner - North   QLD   profit 245231244.21
                                        Dementia Caring   Merrylands - Guildford   NSW   profit 133926565.76
                                  Far West CACP Project Broken Hill and Far West   NSW   profit 100771826.96
                     

In [5]:
# =============================================================================
# STEP 4 — Backfill state, aggregate funding, save
# =============================================================================

# Backfill state for 2019–2022 (raw files lacked state column)
# Use sa3_code → state mapping derived from 2023–2025 rows (100% coverage, no conflicts)
state_map = (
    result[result['state'].notna()][['sa3_code', 'state']]
    .drop_duplicates('sa3_code')
)
result = result.merge(state_map.rename(columns={'state': '_state_fill'}), on='sa3_code', how='left')
result['state'] = result['state'].fillna(result['_state_fill'])
result = result.drop(columns=['_state_fill'])
print(f'State null after backfill: {result["state"].isna().sum()}')

# Aggregate: one row per facility × year — sum funding across all approval IDs
# (raw ACQSC files list multiple approval IDs per facility; funding must be summed)
before = len(result)
result = (
    result
    .groupby(
        ['service_name', 'sa3_code', 'sa3_name', 'state', 'care_type', 'org_type', 'year'],
        dropna=False
    )['funding']
    .sum()
    .reset_index()
)
print(f'Aggregated: {before} → {len(result)} rows ({before - len(result)} approval-ID rows collapsed)')

result['sa3_code'] = result['sa3_code'].astype(str).str.strip()
result.to_csv(OUT, index=False)
print(f'Saved: {OUT}')
print(f'Shape: {result.shape}')
print('Columns:', result.columns.tolist())

# Sanity check
print('\n=== state null by year (post-fix) ===')
print(result.groupby('year')['state'].apply(lambda x: x.isna().sum()).to_string())
print('\n=== Exact duplicates remaining ===', result.duplicated().sum())

State null after backfill: 0
Aggregated: 38481 → 37733 rows (748 approval-ID rows collapsed)
Saved: ../../data/clean/service_funding_by_facility.csv
Shape: (37733, 8)
Columns: ['service_name', 'sa3_code', 'sa3_name', 'state', 'care_type', 'org_type', 'year', 'funding']

=== state null by year (post-fix) ===
year
2019    0
2020    0
2021    0
2022    0
2023    0
2024    0
2025    0

=== Exact duplicates remaining === 0
